In [1]:
#import dependencies
import os.path
os.chdir(".")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from torch.utils.data import DataLoader

import re
import numpy as np
import pandas as pd
import pickle

import transformers, datasets
from transformers.modeling_outputs import TokenClassifierOutput
from transformers.utils.model_parallel_utils import assert_device_map, get_device_map
from transformers.models.esm.modeling_esm import EsmPreTrainedModel, EsmModel
from transformers import AutoTokenizer
from transformers import TrainingArguments, Trainer, set_seed
from transformers import DataCollatorForTokenClassification
from transformers import TrainerCallback

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Union

# for custom DataCollator
from transformers.data.data_collator import DataCollatorMixin
from transformers.tokenization_utils_base import PreTrainedTokenizerBase
from transformers.utils import PaddingStrategy

import peft
from peft import get_peft_config, PeftModel, PeftConfig, inject_adapter_in_model, LoraConfig

from evaluate import load
from datasets import Dataset

from tqdm import tqdm
import random

%matplotlib inline

/home/subinoy/Softwares/anaconda3/envs/finetune/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
ESMs = ["facebook/esm2_t36_3B_UR50D"]
choose_model="esm2_t36_3B"
ptm_type='phosphorylation'

checkpoint = ESMs[0]

choose_gpu=6 

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"]=str(choose_gpu)

In [4]:
def load_fasta_as_dict(fasta_file):
    """
    Load a FASTA file into a dictionary.
    
    Parameters:
    -----------
    fasta_file : str
        Path to the FASTA file
        
    Returns:
    --------
    dict
        Dictionary with FASTA IDs as keys and sequences as values
    """
    fasta_dict = {}
    current_id = None
    current_seq = []
    
    with open(fasta_file, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):               
                if current_id is not None:
                    fasta_dict[current_id] = ''.join(current_seq)              
                
               
                parts = line.split('|')
                if len(parts) >= 2:
                    current_id = parts[1]  
                else:
                    current_id = line[1:]  #
                
                current_seq = []
            else:
                # Add sequence line
                current_seq.append(line)
        
        
        if current_id is not None:
            fasta_dict[current_id] = ''.join(current_seq)
    
    return fasta_dict

In [5]:
class EsmForTokenClassificationCustom(EsmPreTrainedModel):
    _keys_to_ignore_on_load_unexpected = [r"pooler"]
    _keys_to_ignore_on_load_missing = [r"position_ids"]

    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels

        self.esm = EsmModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Sequential(
            nn.Linear(config.hidden_size, 2048),
            nn.LayerNorm(2048),  
            nn.Tanh(),        

            nn.Linear(2048, 1024),
            nn.LayerNorm(1024),  
            nn.Tanh(),

            nn.Linear(1024, 512),
            nn.LayerNorm(512),  
            nn.Tanh(),
            
            nn.Linear(512, 256),
            nn.LayerNorm(256),  
            nn.Tanh(),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),  
            nn.Tanh(),
                   
            nn.Linear(128, 32),  
            nn.LayerNorm(32),    
            nn.Tanh(),

            nn.Linear(32, 16),  
            nn.LayerNorm(16),    
            nn.Tanh(),

            nn.Linear(16, 8),  
            nn.LayerNorm(8),   
            nn.Tanh(),

            nn.Linear(8, 4),  
            nn.LayerNorm(4),  
            nn.Tanh(),
            
            nn.Linear(4, config.num_labels)  # Final projection
        )
        self.init_weights()
        self._initialize_classifier_weights()  # Initialize FFNN weights

    def _initialize_classifier_weights(self):
        """Custom weight initialization for the FFNN classifier."""
        for module in self.classifier:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)  # Xavier Initialization for linear layers
                if module.bias is not None:
                    nn.init.zeros_(module.bias)  # Bias set to zero

    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        head_mask: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        labels: Optional[torch.LongTensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> Union[Tuple, TokenClassifierOutput]:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size, sequence_length)`, *optional*):
            Labels for computing the token classification loss. Indices should be in `[0, ..., config.num_labels - 1]`.
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.esm(
            input_ids,
            attention_mask=attention_mask,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        sequence_output = outputs[0]

        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        # changed to ignore special tokens at the seq start and end 
        # as well as invalid positions (labels -100)
        if labels is not None:
            loss_fct = FocalLoss(gamma=2.0, alpha=torch.tensor([1, focal_loss_power]).to('cuda:0'))  # Adjust weights if needed

            # Mask out positions where attention_mask == 0
            active_loss = attention_mask.view(-1) == 1
            active_logits = logits.view(-1, self.num_labels)

            active_labels = torch.where(active_loss, labels.view(-1), torch.tensor(-100).type_as(labels))

            # Remove masked labels (-100)
            valid_logits = active_logits[active_labels != -100]
            valid_labels = active_labels[active_labels != -100]
            
            valid_labels = valid_labels.type(torch.LongTensor).to('cuda:0')
            
            # Compute Focal Loss
            loss = loss_fct(valid_logits, valid_labels)

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

    
# based on transformers DataCollatorForTokenClassification
@dataclass
class DataCollatorForTokenClassificationESM(DataCollatorMixin):
    """
    Data collator that will dynamically pad the inputs received, as well as the labels.
    Args:
        tokenizer ([`PreTrainedTokenizer`] or [`PreTrainedTokenizerFast`]):
            The tokenizer used for encoding the data.
        padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            - `True` or `'longest'` (default): Pad to the longest sequence in the batch (or no padding if only a single
              sequence is provided).
            - `'max_length'`: Pad to a maximum length specified with the argument `max_length` or to the maximum
              acceptable input length for the model if that argument is not provided.
            - `False` or `'do_not_pad'`: No padding (i.e., can output a batch with sequences of different lengths).
        max_length (`int`, *optional*):
            Maximum length of the returned list and optionally padding length (see above).
        pad_to_multiple_of (`int`, *optional*):
            If set will pad the sequence to a multiple of the provided value.
            This is especially useful to enable the use of Tensor Cores on NVIDIA hardware with compute capability >=
            7.5 (Volta).
        label_pad_token_id (`int`, *optional*, defaults to -100):
            The id to use when padding the labels (-100 will be automatically ignore by PyTorch loss functions).
        return_tensors (`str`):
            The type of Tensor to return. Allowable values are "np", "pt" and "tf".
    """

    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    label_pad_token_id: int = -100
    return_tensors: str = "pt"

    def torch_call(self, features):
        import torch

        label_name = "label" if "label" in features[0].keys() else "labels"
        labels = [feature[label_name] for feature in features] if label_name in features[0].keys() else None

        no_labels_features = [{k: v for k, v in feature.items() if k != label_name} for feature in features]

        batch = self.tokenizer.pad(
            no_labels_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        if labels is None:
            return batch

        sequence_length = batch["input_ids"].shape[1]
        padding_side = self.tokenizer.padding_side

        def to_list(tensor_or_iterable):
            if isinstance(tensor_or_iterable, torch.Tensor):
                return tensor_or_iterable.tolist()
            return list(tensor_or_iterable)

        if padding_side == "right":
            batch[label_name] = [
                # to_list(label) + [self.label_pad_token_id] * (sequence_length - len(label)) for label in labels
                # changed to pad the special tokens at the beginning and end of the sequence
                [self.label_pad_token_id] + to_list(label) + [self.label_pad_token_id] * (sequence_length - len(label)-1) for label in labels
            ]
        else:
            batch[label_name] = [
                [self.label_pad_token_id] * (sequence_length - len(label)) + to_list(label) for label in labels
            ]

        batch[label_name] = torch.tensor(batch[label_name], dtype=torch.float)
        return batch

def _torch_collate_batch(examples, tokenizer, pad_to_multiple_of: Optional[int] = None):
    """Collate `examples` into a batch, using the information in `tokenizer` for padding if necessary."""
    import torch

    # Tensorize if necessary.
    if isinstance(examples[0], (list, tuple, np.ndarray)):
        examples = [torch.tensor(e, dtype=torch.long) for e in examples]

    length_of_first = examples[0].size(0)

    # Check if padding is necessary.

    are_tensors_same_length = all(x.size(0) == length_of_first for x in examples)
    if are_tensors_same_length and (pad_to_multiple_of is None or length_of_first % pad_to_multiple_of == 0):
        return torch.stack(examples, dim=0)

    # If yes, check if we have a `pad_token`.
    if tokenizer._pad_token is None:
        raise ValueError(
            "You are attempting to pad samples but the tokenizer you are using"
            f" ({tokenizer.__class__.__name__}) does not have a pad token."
        )

    # Creating the full tensor and filling it with our data.
    max_length = max(x.size(0) for x in examples)
    if pad_to_multiple_of is not None and (max_length % pad_to_multiple_of != 0):
        max_length = ((max_length // pad_to_multiple_of) + 1) * pad_to_multiple_of
    result = examples[0].new_full([len(examples), max_length], tokenizer.pad_token_id)
    for i, example in enumerate(examples):
        if tokenizer.padding_side == "right":
            result[i, : example.shape[0]] = example
        else:
            result[i, -example.shape[0] :] = example
    return result

def tolist(x):
    if isinstance(x, list):
        return x
    elif hasattr(x, "numpy"):  # Checks for TF tensors without needing the import
        x = x.numpy()
    return x.tolist()

In [6]:
#load ESM2 models
def load_esm_model_classification(checkpoint, num_labels, half_precision, full=False, deepspeed=True):
    
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    
    if half_precision and deepspeed:
        model = EsmForTokenClassificationCustom.from_pretrained(checkpoint, num_labels = num_labels, torch_dtype = torch.float16)
    else:
        model = EsmForTokenClassificationCustom.from_pretrained(checkpoint, num_labels = num_labels)
        
    if full == True:
        return model, tokenizer 
        
    peft_config = LoraConfig(
        r=16, lora_alpha=32, bias="all", target_modules=["query","key","value","dense"]
    )
    
    model = inject_adapter_in_model(peft_config, model)
    
    # Unfreeze the prediction head
    for (param_name, param) in model.classifier.named_parameters():
                param.requires_grad = True  
    
    return model, tokenizer

In [7]:
def load_model(checkpoint, filepath, num_labels=2, mixed = True, full = False, deepspeed=True):
# Creates a new PT5 model and loads the finetuned weights from a file

    # load model
    if "esm" in checkpoint:
        model, tokenizer = load_esm_model_classification(checkpoint, num_labels, mixed, full, deepspeed)
    else:
        model, tokenizer = load_T5_model_classification(checkpoint, num_labels, mixed, full, deepspeed)
    
    # Load the non-frozen parameters from the saved file
    non_frozen_params = torch.load(filepath)

    # Assign the non-frozen parameters to the corresponding parameters of the model
    for param_name, param in model.named_parameters():
        if param_name in non_frozen_params:
            param.data = non_frozen_params[param_name].data

    return tokenizer, model

In [8]:
tokenizer, model = load_model(checkpoint, f"./esm2_t36_3B_finetuned_phosphorylation.pth", num_labels=2)

/home/subinoy/Softwares/anaconda3/envs/finetune/lib/python3.9/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/subinoy/Softwares/anaconda3/envs/finetune/lib/python3.9/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of EsmForTokenClassificationCustom were not initialized from the model checkpoint at facebook/esm2_t36_3B_UR50D and are newly initialized: ['classifier.0.bias', 'classifier.0.weight', 'classifier.1.bias', 'classifier.1.weight', 'classifier.10.bias', 'classifier.10.weight', 'classifier.12.bias', 'classifier.12.weight', 'classifier.13.bias', 'classifier.13.weight', 'classifier.15.bias', 'classifier.15.weight', 'classifier.16.bias', 'classifier.16.weight', 'classifier.18.bias', 'classifier.18.weight', 'classifier.19.bias', 'classifier.19.weight', 'classifi

In [9]:
fasta_dict = load_fasta_as_dict('sample.fasta')
print(f"Loaded {len(fasta_dict)} sequences")

Loaded 3 sequences


In [10]:
sequence_ids = list(fasta_dict.keys())
sequences = list(fasta_dict.values())

In [11]:
# Create dummy labels
def create_labels_for_prediction(sequences):
    labels = []
    for sequence in sequences:
        seq_labels = []
        for residue in sequence:
            if residue in ['S', 'T', 'Y']:
                seq_labels.append(0)  # Placeholder
            else:
                seq_labels.append(-100)  # Ignore
        labels.append(seq_labels)
    return labels

dummy_labels = create_labels_for_prediction(sequences)

In [12]:
len(dummy_labels[-1])

2460

In [13]:
WINDOW_SIZE = 1022
STEP = WINDOW_SIZE

In [14]:
def split_sequence(sequence,
                   window_size=WINDOW_SIZE,
                   step=STEP):

    chunks = []

    for start in range(0, len(sequence), step):

        end = start + window_size

        chunk = sequence[start:end]

        if len(chunk) == 0:
            continue

        chunks.append((start, chunk))

        if end >= len(sequence):
            break

    return chunks


In [15]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

EsmForTokenClassificationCustom(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 2560, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 2560, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-35): 36 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): lora.Linear(
                (base_layer): Linear(in_features=2560, out_features=2560, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2560, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_e

In [16]:
predictions_dict = {}
output_file = 'phosphorylation_predictions_STY.txt'

In [17]:
with open(output_file, 'w') as f:

    f.write("Sequence_ID\tPosition\tResidue\tPhosphorylation\n")

    with torch.no_grad():

        for seq_id, sequence in tqdm(
                zip(sequence_ids, sequences),
                total=len(sequences)):

            seq_len = len(sequence)

            # ====================================================
            # Initialize predictions
            # ====================================================

            final_predictions = [-100] * seq_len

            # ====================================================
            # Create chunks
            # ====================================================

            chunks = split_sequence(sequence)

            # ====================================================
            # Process chunks
            # ====================================================

            for chunk_start, chunk_seq in chunks:

                tokenized = tokenizer(
                    chunk_seq,
                    return_tensors="pt",
                    truncation=True,
                    max_length=1024
                )

                input_ids = tokenized["input_ids"].to(device)
                attention_mask = tokenized["attention_mask"].to(device)

                outputs = model.float()(
                    input_ids,
                    attention_mask=attention_mask
                )

                # Class predictions (0 or 1)
                preds = outputs.logits.argmax(dim=-1)

                preds = preds[0].cpu().numpy()

                # ====================================================
                # Remove special tokens
                # ====================================================

                if ("esm" in checkpoint) or ("ProstT5" in checkpoint):

                    preds = preds[1:len(chunk_seq)+1]

                else:

                    preds = preds[:len(chunk_seq)]

                # ====================================================
                # Store predictions
                # ====================================================

                for i, pred in enumerate(preds):

                    global_pos = chunk_start + i

                    if global_pos >= seq_len:
                        continue

                    residue = sequence[global_pos]

                    # Only predict S/T/Y residues
                    if residue in ['S', 'T', 'Y']:

                        final_predictions[global_pos] = int(pred)

            # ====================================================
            # Save predictions dictionary
            # ====================================================

            predictions_dict[seq_id] = final_predictions

            # ====================================================
            # Write output file
            # ====================================================

            for idx, (residue, pred) in enumerate(
                    zip(sequence, final_predictions),
                    start=1):

                # Only report S/T/Y residues
                if residue not in ['S', 'T', 'Y']:
                    continue

                is_phosphorylated = (
                    "True" if pred == 1 else "False"
                )

                f.write(
                    f"{seq_id}\t"
                    f"{idx}\t"
                    f"{residue}\t"
                    f"{is_phosphorylated}\n"
                )

            f.write("\n")

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:01<00:00,  1.77it/s]


In [18]:
pickle_file = 'phosphorylation_predictions_all.pkl'

with open(pickle_file, 'wb') as f:
    pickle.dump(predictions_dict, f)

print(f"\nPredictions for S/T/Y residues saved to {output_file}")
print(f"All predictions saved to {pickle_file}")


Predictions for S/T/Y residues saved to phosphorylation_predictions_STY.txt
All predictions saved to phosphorylation_predictions_all.pkl


In [19]:
len(predictions_dict['FASTA_1'])

238

In [20]:
predictions_dict['FASTA_1']

[-100,
 -100,
 -100,
 -100,
 1,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 1,
 1,
 -100,
 0,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 0,
 -100,
 -100,
 0,
 -100,
 0,
 -100,
 1,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 0,
 -100,
 0,
 -100,
 -100,
 0,
 -100,
 -100,
 1,
 1,
 -100,
 -100,
 -100,
 1,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 1,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 1,
 1,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 0,
 -100,
 0,
 0,
 0,
 -100,
 -100,
 0,
 -100,
 0,
 -100,
 0,
 -100,
 -100,
 -100,
 0,
 0,
 -100,
 0,
 -100,
 -100,
 0,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 0,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 0,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100

In [21]:
len(predictions_dict['FASTA_1'])

238

In [22]:
len(fasta_dict['FASTA_1'])

238